# Replay with no Drive and no key: the hallucination table and one fixture, offline (Phase 5 task I, DEC-37)

Needs no Google Drive, no API key, no GPU and no patient data. It installs the package from the public repository, gets the replay bundle (3 MB, an asset of the repository's release) and then:
- **rebuilds the hallucination table** for the six language models on the five fixtures and the 150 synthetic scenarios, from the saved answers and the verifier's caches, and checks every rate against the reported one (`python -m strokecopilot.pipeline.replay run`);
- **runs one hand-written fixture end to end**: its vision output and triage note, one model's saved answer, and that answer scored claim by claim by the verifier.

No model is called: a judge verdict or a retrieval the caches lack is an error, never an API call. The bundle holds the fixtures and the synthetic scenarios only. The 20 real cases per model are not in it, so the pooled rates printed here differ a little from the README's headline rates, which include them. The logic is in `src/strokecopilot/pipeline/replay.py`.

**Run it:** Runtime → Run all. A CPU runtime is enough and it takes under a minute. On your own machine, with the package installed (`pip install -e ".[dev]"`), it uses that install, and a local copy of the bundle if there is one. **Done when** section 6 prints `OK:`.

The outputs stored below are from a run on 22 Nov 2026.

## 1. Settings

In [1]:
REPO_REF = 'main'                             # branch, tag or commit of the public repository
FIXTURE = 'fx02_thrombectomy_large_mismatch'  # any of the five in fixtures/
MODEL = 'gemma-3-4b-tuned'                    # or claude-sonnet-5-5, qwen3.5-27b, gemma-3-27b, medgemma-27b-text, gemma-3-4b
LOCAL_BUNDLES = ('replay_bundle.tar.gz', 'runs/replay_bundle.tar.gz', '../runs/replay_bundle.tar.gz')

## 2. Install the package
In Colab, from the public repository at `REPO_REF`; no token is needed. Anywhere else, the package already installed is used. The replay needs only the base package and numpy.

In [2]:
import shutil, subprocess, sys

if 'google.colab' in sys.modules:
    url = f'strokecopilot @ git+https://github.com/parsa-hke/strokecopilot.git@{REPO_REF}'
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', url], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr[-800:])
    del url, result
from strokecopilot.pipeline import replay
print(f'Package ready. The bundle is the asset {replay.BUNDLE_NAME} of release {replay.RELEASE_TAG}.')

Package ready. The bundle is the asset replay_bundle.tar.gz of release v1.0.1.


## 3. Get the replay bundle
A local copy if there is one; otherwise the release asset is downloaded. Either way its sha256 must be the one recorded in the code.

In [3]:
import tempfile
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix='replay_'))  # a temporary folder: nothing is written to the repository or to Drive
bundle = replay.fetch(WORK / replay.BUNDLE_NAME, local=LOCAL_BUNDLES)

Local copy ../runs/replay_bundle.tar.gz: 3.2 MB, sha256 as recorded


## 4. Rebuild the hallucination table
The README's command, `python -m strokecopilot.pipeline.replay run --bundle replay_bundle.tar.gz`. It rescores every model and arm from the cached verdicts (20 cells), checks each rate against the reported one, then prints each model's pooled rate with its 95% interval over cases. These are verifier rates: the human check found that the verifier both over-flags and misses errors (README, *Verifier validity*).

In [4]:
command = [sys.executable, '-m', 'strokecopilot.pipeline.replay', 'run', '--bundle', str(bundle), '--workdir', str(WORK / 'cells')]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout.rstrip())
assert 'OK: every cell matches its reported rates.' in result.stdout, result.stderr[-800:]

claude-sonnet-5-5/fixtures_norulebook      hallucination  23.0%  content  22.5%  ok
claude-sonnet-5-5/fixtures_rulebook        hallucination   5.6%  content   4.8%  ok
claude-sonnet-5-5/scenarios_norulebook     hallucination  19.8%  content  19.6%  ok
claude-sonnet-5-5/scenarios_rulebook       hallucination   4.0%  content   3.6%  ok
gemma-3-27b/fixtures_norulebook            hallucination  24.5%  content  24.5%  ok
gemma-3-27b/fixtures_rulebook              hallucination  13.8%  content   8.5%  ok
gemma-3-27b/scenarios_norulebook           hallucination  31.8%  content  28.4%  ok
gemma-3-27b/scenarios_rulebook             hallucination  18.2%  content  14.4%  ok
gemma-3-4b-tuned/fixtures_rulebook         hallucination  15.5%  content  12.7%  ok
gemma-3-4b-tuned/scenarios_rulebook        hallucination  13.3%  content  12.4%  ok
gemma-3-4b/fixtures_rulebook               hallucination  37.5%  content  33.7%  ok
gemma-3-4b/scenarios_rulebook              hallucination  41.5%  content  34

## 5. One fixture, end to end
`replay.one_case` copies this fixture's saved files and its cell's caches to a folder of their own and scores them with `end_to_end.rescore`, as section 4 did for every case. First, what the reasoner was given: the vision output (hand-written, in the shape Vision A and Vision B produce) and the triage note.

In [5]:
import json

one = replay.one_case(WORK / 'cells', FIXTURE, MODEL)
print(json.dumps(one['vision'], indent=1))
print()
print(one['note'])

{
 "schema_version": "0.1",
 "case_id": "fixture-fx02",
 "model_version": "fixture-handwritten",
 "haemorrhage": {
  "present": false,
  "confidence": 0.04
 },
 "core_ml": 18.0,
 "penumbra_ml": 95.0,
 "mismatch_ratio": 5.28,
 "mismatch_volume_ml": 77.0,
 "occlusion_site": "M1",
 "aspects": 8,
 "confidence": {
  "core": 0.86,
  "penumbra": 0.88
 },
 "warnings": []
}

ED STROKE TRIAGE NOTE
64-year-old man. Last seen well by a colleague at 08:00; found at 15:30 slumped at his desk with left-sided weakness and left gaze neglect. Onset not witnessed.
Time from last known well to imaging: 9 h 0 min.
NIHSS on arrival: 16.
Pre-stroke mRS: 0 (fully independent).
Medications: atorvastatin. No anticoagulant use.
History: hyperlipidaemia, current smoker (20 pack-years). No recent surgery or bleeding.
Observations: BP 154/88 mmHg, capillary glucose 6.1 mmol/L.



The answer `MODEL` gave when the experiment ran, with the whole rulebook in its prompt (DEC-19). It is read from the bundle; no model is called now.

In [6]:
print(json.dumps(one['answer'], indent=1, ensure_ascii=False) if one['answer'] else one['error'])

{
 "case_id": "fixture-fx02",
 "model": "gemma-3-4b-tuned",
 "prompt_version": "reasoner_v3",
 "thrombolysis": {
  "recommend": false,
  "rule_ids": [
   "AHA2026-4.6.3-wake-up-perfusion",
   "AHA2026-4.6.3-lvo-4.5-24h",
   "AHA2026-4.6.1-within-4.5h"
  ],
  "blocking_factors": [
   "Time from last known well to imaging 9 h 0 min is beyond 4.5 hours."
  ]
 },
 "thrombectomy": {
  "recommend": false,
  "rule_ids": [
   "AHA2026-4.7.2-6-24h-aspects-6-10",
   "AHA2026-4.7.2-6-24h-aspects-3-5"
  ],
  "blocking_factors": [
   "ASPECTS 8 is below 6."
  ]
 },
 "rationale": [
  {
   "claim": "ASPECTS 8 is below 6.",
   "sources": [
    {
     "type": "vision",
     "field": "aspects"
    },
    {
     "type": "passage",
     "id": "AHA2026-4.7.2-6-24h-aspects-6-10"
    }
   ],
   "derived": true
  },
  {
   "claim": "Time from last known well to imaging 9 h 0 min is beyond 4.5 hours.",
   "sources": [
    {
     "type": "note",
     "quote": "Time from last known well to imaging: 9 h 0 min."
 

The verifier's scoring of that answer: each decision against the fixture's expected one, then every claim, traced to its source (`ok`) or flagged (`FLAG`, with the hallucination sub-type and the reason).

In [7]:
print('\n'.join(replay.decision_lines(one)))
print()
print('\n'.join(replay.claim_lines(one)))

thrombolysis: answered no, expected no, right
thrombectomy: answered no, expected yes, WRONG

FLAG rationale[0].0                       comparison  ASPECTS 8 is below the threshold of 6
     -> faulty_derivation (critical): With the true values, 8 < 6 is false.
ok   rationale[1].0                       comparison  Last known well to imaging time of 9 h 0 min is beyond 4.5 hours
FLAG rationale[2].0                       rule        The patient does not meet the criteria for thrombectomy
     -> misapplied_rule (critical): A passage states the rule, but its scope does not fit this patient.
ok   rationale[3].0                       comparison  Last known well to imaging time of 9 h 0 min is beyond 4.5 hours
ok   thrombolysis.blocking_factors[0].0   comparison  Last known well to imaging time of 9 h 0 min is beyond 4.5 hours
FLAG thrombectomy.blocking_factors[0].0   comparison  ASPECTS 8 is below the threshold of 6
     -> faulty_derivation (critical): With the true values, 8 < 6 is false.

## 6. Summary

In [8]:
critical = sum(bool(f['critical']) for f in one['flagged'])
print(f"OK: the replay matched every reported rate. {FIXTURE[:4]} answered by {MODEL}: {len(one['flagged'])} of {len(one['claims'])} "
      f"claims flagged, {critical} critical. No model was called and no key was read.")
shutil.rmtree(WORK, ignore_errors=True)  # the temporary folder

OK: the replay matched every reported rate. fx02 answered by gemma-3-4b-tuned: 3 of 14 claims flagged, 3 critical. No model was called and no key was read.
